# 04_3 — Physics-Informed Machine Learning (PIML) & Locally Weighted Conformal Prediction

This notebook showcases the new **Physics-Informed Machine Learning (`quick_pp.machine_learning`)** pipeline:
1. **Domain Rock-Physics Feature Engineering (`generate_fe_features`)**: Apparent matrix density (`RHOMA_APP`), Acoustic Impedance (`AI` with Gardner's relation fallback), Burke's M-N lithology parameters (`M_LITH`, `N_LITH`), `TIGHT_FLAG`, and `COAL_FLAG`.
2. **Locally Weighted Split & Multi-Output Conformal Prediction (`ConformalRegressor`, `ConformalMultiOutputRegressor`)**: Calibrated heteroscedastic `P10`–`P90` intervals scaled by local ensemble tree disagreement $\hat{\sigma}(x)$ with small-sample quantile monotonicity guards.
3. **Physics-Informed Sample Weights, Loss & Volumetric Closure (`compute_physics_sample_weights`, `physics_informed_loss`, `EnforceMineralSum`, `enforce_physical_constraints`)**.

In [ ]:
import sys
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
for _cand in (Path.cwd(), Path.cwd().parent):
    if (_cand / "quick_pp" / "__init__.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quick_pp import (
    ConformalRegressor,
    ConformalMultiOutputRegressor,
    EnforceMineralSum,
    enforce_physical_constraints,
    physics_informed_loss,
)
from quick_pp.machine_learning import (
    generate_fe_features,
    apparent_matrix_density,
    acoustic_impedance_feature,
    mn_lithology_parameters,
    compute_physics_sample_weights,
    predict_with_uncertainty,
)

## Part A: Domain Rock-Physics Feature Engineering (`generate_fe_features`)

In [ ]:
data_path = Path("../quick_pp/machine_learning/mock_data/mock_data.parquet")
if not data_path.exists():
    data_path = Path("quick_pp/machine_learning/mock_data/mock_data.parquet")

raw_df = pd.read_parquet(data_path).sort_values("DEPTH").reset_index(drop=True)
fe_df = generate_fe_features(raw_df)

physics_cols = ["DEPTH", "GR", "RHOB", "NPHI", "DT", "RHOMA_APP", "AI", "M_LITH", "N_LITH", "TIGHT_FLAG", "COAL_FLAG", "ROCK_FLAG"]
display(fe_df[physics_cols].head(8).round(4))

## Part B: Locally Weighted `ConformalRegressor` & Small-Sample Monotonicity Across $\alpha$

In [ ]:
# Prepare targets with mineral volumes + porosity + saturation
fe_df["PHIT"] = fe_df["PHIE"].clip(0.01, 0.38)
fe_df["SWT"] = fe_df["SW"].clip(0.05, 1.0)
fe_df["VCLAY"] = ((fe_df["GR"] - 15.0) / 115.0).clip(0.0, 0.85)
fe_df["VSAND"] = np.clip((1.0 - fe_df["PHIT"] - fe_df["VCLAY"]) * 0.72, 0.0, 1.0)
fe_df["VSILT"] = np.clip(1.0 - fe_df["PHIT"] - fe_df["VCLAY"] - fe_df["VSAND"], 0.0, 1.0)

# Inject a tight streak (PHIT < 0.02, TIGHT_FLAG = 1) to verify physics-informed clamping
tight_slice = fe_df.index[40:55]
fe_df.loc[tight_slice, "TIGHT_FLAG"] = 1.0
fe_df.loc[tight_slice, "PHIT"] = 0.015
fe_df.loc[tight_slice, "PHIE"] = 0.008
fe_df.loc[tight_slice, "SWT"] = 1.0

features = ["GR", "RHOB", "NPHI", "RT", "DPHI", "RHOMA_APP", "AI", "M_LITH", "N_LITH", "TIGHT_FLAG", "COAL_FLAG"]
targets = ["PHIT", "PHIE", "SWT", "VCLAY", "VSAND", "VSILT"]

valid_df = fe_df.dropna(subset=features + targets).reset_index(drop=True)
X = valid_df[features]
y = valid_df[targets]

# Train/test split
n_train = int(0.70 * len(valid_df))
X_tr, X_te = X.iloc[:n_train], X.iloc[n_train:]
y_tr, y_te = y.iloc[:n_train], y.iloc[n_train:]

# Fit single-output ConformalRegressor and verify strict interval widening as alpha decreases (80% vs 90% vs 95% confidence)
cr = ConformalRegressor(alpha=0.20, random_state=42).fit(X_tr, y_tr["PHIT"])
for alpha_val in (0.30, 0.20, 0.10, 0.05):
    pred, low, high = cr.predict_interval(X_te, alpha=alpha_val)
    mean_width = float(np.mean(high - low))
    cov = float(np.mean((y_te["PHIT"] >= low) & (y_te["PHIT"] <= high)))
    print(f"alpha = {alpha_val:.2f} ({int((1-alpha_val)*100)}% CI) -> Mean Interval Width = {mean_width:.4f} v/v | Empirical Coverage = {cov:.1%}")

## Part C: Multi-Output Conformal PIML with `EnforceMineralSum` & `enforce_physical_constraints`

In [ ]:
# Compute physics-informed sample weights down-weighting any noisy training labels that violate closure
sample_weights = compute_physics_sample_weights(X_tr, y_tr)
print(f"Physics Sample Weights range: [{sample_weights.min():.3f}, {sample_weights.max():.3f}]")

# Fit ConformalMultiOutputRegressor with enforce_physics=True
multi_conf = ConformalMultiOutputRegressor(
    alpha=0.20,
    cv=4,
    enforce_physics=True,
    target_names=targets,
    random_state=42,
)
multi_conf.fit(X_tr, y_tr, sample_weight=sample_weights)

y_pred, y_p10, y_p90 = multi_conf.predict_interval(X_te, alpha=0.20)
res_df = pd.DataFrame(y_pred, columns=targets, index=X_te.index)
for i, t in enumerate(targets):
    res_df[f"{t}_P10"] = y_p10[:, i]
    res_df[f"{t}_P90"] = y_p90[:, i]

# Verify physical constraints:
# 1. Volumetric closure: VCLAY + VSAND + VSILT + PHIT == 1.0
closure = res_df["VCLAY"] + res_df["VSAND"] + res_df["VSILT"] + res_df["PHIT"]
# 2. Porosity hierarchy: PHIE <= PHIT
poro_violation = np.maximum(0.0, res_df["PHIE"] - res_df["PHIT"]).max()
# 3. Monotonicity: P10 <= Pred <= P90
mono_ok = bool(np.all((res_df["PHIT_P10"] <= res_df["PHIT"] + 1e-9) & (res_df["PHIT"] <= res_df["PHIT_P90"] + 1e-9)))

print(f"Max Volumetric Closure Deviation |∑V + PHIT - 1| : {np.max(np.abs(closure - 1.0)):.2e}")
print(f"Max Porosity Hierarchy Violation (PHIE > PHIT)    : {poro_violation:.2e}")
print(f"P10 <= Pred <= P90 Monotonicity Guaranteed        : {mono_ok}")
print(f"Physics-Informed Composite Loss on Test Set       : {physics_informed_loss(y_te, res_df[targets], target_names=targets, X=X_te):.5f}")

display(res_df.head(8).round(4))